# Wind turbine outage: wind speed prediction pipeline

1. Load data -> `ds_train`, `ds_test`, `ds_result`
2. Power curves
3. Fit candidate models on `ds_train`
4. Evaluate every model on `ds_test`
5. Pick the best model
6. Retrain it on train + test, predict `ds_result`
7. Energy loss and submission export

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from get_dataset import get_data, target_turbines, reference_turbines
from power_curves import POWER_CURVE, energy_kwh, powercurve
from models import neighbour_mean, neighbour_mean_params, neighbour_mean_sector, neighbour_mean_sector_params
from models import ridge_all_references, ridge_all_references_params
from models import ridge_all_references_sector, ridge_all_references_sector_params
from evaluation import evaluate, select_best

## 1. Load data

In [2]:
ds_train, ds_test, ds_result = get_data(
    data_dir="data",
    test_start="2026-09-04",
    test_end=None,            # None = to the end of the data; "2026-09-09" = README shown test period
    result_period=None,       # None = auto-detect the outage; or ("2026-09-01 00:00", "2026-09-03 23:50")
    use_non_complete=False,   # False = train only on rows where every turbine has a wind speed
)

for name, ds in [("train", ds_train), ("test", ds_test), ("result", ds_result)]:
    print(f"{name:7s} {ds.index[0]} -> {ds.index[-1]}  {len(ds):5d} rows")
print("targets:   ", target_turbines(ds_train))
print("references:", reference_turbines(ds_train))

park H: 288 timestamps without a valid wind direction
park O: 291 timestamps without a valid wind direction
park T: 720 timestamps without a valid wind direction
samples per park (rows) and sector (columns):
    0    1    2     3    4    5    6     7     8     9     10   11
H  499  357  520   708  936  483  403  1005  2649  2532  2072  796
O  214  306  648   908  830  308  626  1306  2584  2765  1649  813
T  315  402  683  1022  656  436  640  1644  2069  2579  1454  628
train   2026-07-01 00:00:00 -> 2026-08-31 23:40:00   7622 rows
test    2026-09-04 00:00:00 -> 2026-09-30 23:50:00   3888 rows
result  2026-09-01 00:00:00 -> 2026-09-03 23:50:00    432 rows
targets:    ['T1', 'T2', 'T3']
references: ['H1', 'H2', 'H3', 'H4', 'H5', 'O1', 'O2']


In [3]:
ds_train.head()

turbine                     H1                                     \
variable            wind_speed wind_direction ambient_temperature   
timestamp                                                           
2026-07-01 00:00:00   5.631282       51.43747            16.91814   
2026-07-01 00:20:00   4.966313       99.29465            17.21615   
2026-07-01 00:30:00   4.984415       89.38460            17.40417   
2026-07-01 00:40:00   5.017492       83.53693            17.29220   
2026-07-01 00:50:00   5.199528       82.57381            17.39974   

turbine                                                                    \
variable            air_pressure wind_speed_stddev   park_dir n_dir_valid   
timestamp                                                                   
2026-07-01 00:00:00     993.5333          0.846214  52.443321           5   
2026-07-01 00:20:00     993.6462          0.799924  91.047863           5   
2026-07-01 00:30:00     993.3114          0.639666  83.197480           5   
2026-07-01 00:40:00     993.2694          0.971072  78.805324           5   
2026-07-01 00:50:00     993.1996          1.024312  74.815749           5   

turbine                            H2                 ...     T2         T3  \
variable            sector wind_speed wind_direction  ... sector wind_speed   
timestamp                                             ...                     
2026-07-01 00:00:00      2   6.253799       50.91308  ...      4   4.829981   
2026-07-01 00:20:00      3   5.158716       91.72142  ...      4   2.448443   
2026-07-01 00:30:00      3   5.016369       83.72660  ...      7   1.628271   
2026-07-01 00:40:00      3   5.426286       80.75471  ...      8   0.920418   
2026-07-01 00:50:00      2   5.067235       77.03947  ...      9   0.887650   

turbine                                                              \
variable            wind_direction ambient_temperature air_pressure   
timestamp                                                             
2026-07-01 00:00:00       79.34425            17.46686     1003.363   
2026-07-01 00:20:00      123.71370            17.96162     1003.504   
2026-07-01 00:30:00      198.56650            17.93876     1003.641   
2026-07-01 00:40:00      196.12830            17.88211     1003.583   
2026-07-01 00:50:00      231.42650            18.17443     1003.435   

turbine                                                              complete  
variable            wind_speed_stddev    park_dir n_dir_valid sector           
timestamp                                                                      
2026-07-01 00:00:00          0.807648  105.032879           3      4     True  
2026-07-01 00:20:00          0.698420  133.831463           3      4     True  
2026-07-01 00:30:00          0.480172  221.381710           3      7     True  
2026-07-01 00:40:00          0.404659  242.278713           3      8     True  
2026-07-01 00:50:00          0.394133  269.539506           3      9     True  

[5 rows x 81 columns]

In [4]:
# share of missing wind speed per turbine and set
pd.DataFrame({
    name: ds.xs("wind_speed", axis=1, level=1).isna().mean()
    for name, ds in [("train", ds_train), ("test", ds_test), ("result", ds_result)]
}).round(3)

,train,test,result
turbine,,,
H1,0.0,0.012,0.0
H2,0.0,0.006,0.0
H3,0.0,0.009,0.0
H4,0.0,0.010,0.0
H5,0.0,0.008,0.0
O1,0.0,0.008,0.0
O2,0.0,0.005,0.0
T1,0.0,0.009,1.0
T2,0.0,0.005,1.0


## 2. Power curves

In [5]:
# one curve for T1, T2 and T3 (identical files): wind speed (m/s) in, power (kW) out,
# on the straight line between the two neighbouring points of the 0.5 m/s table
powercurve(7.25)

np.float64(1887.0)

In [6]:
# 0 below cut-in, 4500 kW rated, ramp-down above 20 m/s, 0 above 26 m/s (cut-out)
wind = [2.5, 3.0, 3.25, 7.25, 12.0, 21.0, 26.0, 26.5]
pd.DataFrame({"power_kw": powercurve(wind)}, index=pd.Index(wind, name="wind_speed"))

,power_kw
wind_speed,
2.50,0.0
3.00,31.0
3.25,76.5
7.25,1887.0
12.00,4500.0
21.00,4307.0
26.00,2529.0
26.50,0.0


## 3. Models

**neighbour_mean** - same method as `4_Wind park T - all turbines calculation.xlsx`:
1. Average the wind speed of the 7 neighbouring turbines (H1-H5, O1, O2) at every 10-minute step.
2. Fit one straight line per target turbine on the training data: `T = intercept + slope * average`.
3. Predict with that line, never below 0 m/s.

**neighbour_mean_sector** - the same straight line, but one per wind direction sector:
1. Each 10-minute step gets the 30° sector of park H's mean wind direction (`sector` from `get_data`).
2. Fit `T = intercept + slope * average` separately on the training rows of each sector.
3. Predict each step with the line of its sector; unknown or sparse sectors (< 150 rows) use the global line.

**ridge_all_references** - every reference turbine with its own weight instead of one average:
1. Fit per target turbine `T = intercept + w_H1 * H1 + ... + w_O2 * O2` on the training data
   (Ridge regression: least squares with a small penalty on large weights).
2. Fill missing reference values first: straight line in time, otherwise the training median.
3. Predict with those weights, never below 0 m/s.

**ridge_all_references_sector** - the same weights, but fitted separately per wind direction sector
(like `neighbour_mean_sector`); unknown or sparse sectors use the global weights.

In [7]:
# every model is a function model(ds_train, ds) -> predicted wind speed of T1-T3 for ds
models = {
    "neighbour_mean": neighbour_mean,
    "neighbour_mean_sector": neighbour_mean_sector,
    "ridge_all_references": ridge_all_references,
    "ridge_all_references_sector": ridge_all_references_sector,
}

# fitted line per target turbine: T = intercept + slope * mean(H1..H5, O1, O2)
neighbour_mean_params(ds_train)

,slope,intercept,r2,corr,n
T1,0.730321,0.955756,0.540928,0.735478,7622
T2,0.842195,0.584915,0.652672,0.807881,7622
T3,0.745922,0.815914,0.614408,0.783842,7622


In [8]:
# neighbour_mean_sector: the same line, fitted separately per 30° wind direction sector of park H
# (sector 0 = 345-15°, 1 = 15-45°, ...); prediction uses the line of each time point's sector
neighbour_mean_sector_params(ds_train).round(3)

slope  intercept     r2   corr     n
sector target                                      
0      T1      0.755      0.895  0.495  0.704   393
       T2      0.637      1.109  0.432  0.657   393
       T3      0.627      1.182  0.468  0.684   393
1      T1      0.560      1.567  0.379  0.615   233
       T2      0.506      1.700  0.320  0.566   233
       T3      0.513      1.623  0.324  0.569   233
2      T1      0.892      0.406  0.653  0.808   363
       T2      0.590      1.315  0.416  0.645   363
       T3      0.875      0.332  0.669  0.818   363
3      T1      0.939      0.437  0.732  0.856   509
       T2      0.724      1.071  0.647  0.804   509
       T3      0.804      0.778  0.714  0.845   509
4      T1      0.879      0.783  0.778  0.882   550
       T2      0.768      1.117  0.700  0.837   550
       T3      0.574      1.441  0.652  0.807   550
5      T1      0.668      1.552  0.521  0.722   314
       T2      0.823      0.996  0.616  0.785   314
       T3      0.578      1.725  0.471  0.686   314
6      T1      0.523      1.760  0.430  0.656   223
       T2      0.733      1.020  0.653  0.808   223
       T3      0.661      1.412  0.562  0.750   223
7      T1      0.700      0.883  0.403  0.635   447
       T2      0.942      0.156  0.669  0.818   447
       T3      0.840      0.555  0.606  0.779   447
8      T1      0.834      0.210  0.564  0.751  1110
       T2      0.884      0.413  0.641  0.801  1110
       T3      0.813      0.476  0.605  0.778  1110
9      T1      0.585      1.278  0.425  0.652  1427
       T2      0.878      0.690  0.624  0.790  1427
       T3      0.745      0.875  0.571  0.756  1427
10     T1      0.754      0.962  0.591  0.769  1442
       T2      0.816      0.713  0.663  0.814  1442
       T3      0.734      0.823  0.603  0.777  1442
11     T1      0.965     -0.076  0.532  0.729   611
       T2      0.860      0.176  0.535  0.732   611
       T3      0.804      0.259  0.478  0.691   611

In [9]:
# ridge_all_references: one weight per reference turbine and target; r2 = share of the target's
# variation the formula explains on the training rows
ridge_all_references_params(ds_train).round(3)

,H1,H2,H3,H4,H5,O1,O2,intercept,r2,n
T1,0.355,0.095,0.022,-0.081,-0.029,0.427,-0.024,0.800,0.582,7622
T2,0.015,0.038,0.164,0.045,0.115,0.194,0.242,0.677,0.683,7622
T3,0.064,0.111,0.004,0.087,0.099,0.256,0.127,0.806,0.638,7622


In [10]:
# ridge_all_references_sector: the same weights, fitted separately per 30° wind direction sector of park H
ridge_all_references_sector_params(ds_train).round(3)

H1     H2     H3     H4     H5     O1     O2  intercept  \
sector target                                                               
0      T1      0.333  0.309 -0.096  0.255 -0.449  0.064  0.436      0.859   
       T2      0.395 -0.009 -0.117  0.343 -0.267  0.000  0.370      1.043   
       T3      0.170  0.358  0.074  0.018 -0.254 -0.049  0.335      1.260   
1      T1     -0.055  0.109 -0.100 -0.022  0.316 -0.084  0.416      1.579   
       T2      0.008 -0.095  0.030  0.045  0.209 -0.078  0.452      1.616   
       T3     -0.084  0.163 -0.004 -0.115  0.241 -0.248  0.597      1.681   
2      T1      0.015  0.248  0.159  0.070  0.011  0.497 -0.066      0.126   
       T2     -0.178 -0.105  0.003  0.662  0.029  0.457 -0.151      0.755   
       T3      0.078  0.306  0.165 -0.019 -0.046  0.389  0.040      0.121   
3      T1      0.024  0.026 -0.022 -0.014  0.373  0.531 -0.040      0.431   
       T2     -0.007  0.281 -0.071 -0.036  0.187  0.276  0.106      0.958   
       T3     -0.087  0.044 -0.045  0.094  0.326  0.374  0.048      0.796   
4      T1      0.412  0.029  0.106 -0.065  0.092  0.187  0.077      0.839   
       T2      0.524 -0.085  0.043 -0.117  0.098  0.295 -0.038      1.108   
       T3      0.218  0.043 -0.120  0.042  0.167  0.205  0.024      1.340   
5      T1      0.024  0.289  0.042 -0.224  0.168  0.170  0.206      1.521   
       T2     -0.131  0.385  0.149 -0.230  0.193  0.024  0.402      1.197   
       T3      0.171  0.228  0.068 -0.170 -0.013 -0.084  0.388      1.784   
6      T1      0.583 -0.037 -0.109 -0.063 -0.110  0.146  0.129      1.740   
       T2      0.314  0.345 -0.017 -0.153 -0.160  0.196  0.216      1.056   
       T3      0.446  0.251  0.022 -0.185 -0.239  0.113  0.254      1.497   
7      T1      0.265  0.490 -0.215 -0.185 -0.154  0.228  0.241      0.996   
       T2      0.310  0.192  0.040 -0.238  0.080  0.080  0.398      0.376   
       T3      0.221  0.183 -0.046 -0.045  0.047  0.136  0.319      0.634   
8      T1      0.063  0.262  0.031  0.092 -0.010  0.379  0.015      0.227   
       T2      0.009  0.046  0.176  0.050  0.097  0.187  0.280      0.552   
       T3      0.110  0.131  0.119 -0.071  0.091  0.292  0.115      0.534   
9      T1      0.083  0.132 -0.076  0.243 -0.099  0.357 -0.042      1.342   
       T2      0.110 -0.065  0.173  0.213 -0.032  0.163  0.275      0.808   
       T3      0.002  0.097  0.030  0.156  0.051  0.240  0.150      0.937   
10     T1      0.115 -0.149  0.159  0.197  0.015  0.328  0.083      0.817   
       T2     -0.009  0.050  0.111  0.286 -0.046  0.290  0.119      0.688   
       T3     -0.032 -0.007  0.075  0.388 -0.064  0.270  0.085      0.799   
11     T1      0.157  0.076  0.170 -0.147  0.061  0.701 -0.105      0.089   
       T2      0.243  0.097  0.145 -0.210  0.023  0.528 -0.007      0.352   
       T3      0.268  0.012  0.081 -0.189  0.113  0.567 -0.092      0.403   

                  r2     n  
sector target               
0      T1      0.606   393  
       T2      0.502   393  
       T3      0.508   393  
1      T1      0.461   233  
       T2      0.436   233  
       T3      0.437   233  
2      T1      0.677   363  
       T2      0.493   363  
       T3      0.692   363  
3      T1      0.761   509  
       T2      0.672   509  
       T3      0.741   509  
4      T1      0.790   550  
       T2      0.730   550  
       T3      0.667   550  
5      T1      0.569   314  
       T2      0.665   314  
       T3      0.511   314  
6      T1      0.462   223  
       T2      0.683   223  
       T3      0.595   223  
7      T1      0.485   447  
       T2      0.722   447  
       T3      0.653   447  
8      T1      0.583  1110  
       T2      0.678  1110  
       T3      0.633  1110  
9      T1      0.484  1427  
       T2      0.658  1427  
       T3      0.603  1427  
10     T1      0.645  1442  
       T2      0.700  1442  
       T3      0.647  1442  
11     T1      0.635   611  
       T2      0.636   611  
       T3      0.5

## 4. Evaluate on the test set

In [11]:
PRICE_EUR_PER_MWH = 80   # same price as the Summary sheet of the Excel calculation

# target turbines are hidden before predicting, so the test period looks exactly like an outage
scores = pd.concat({name: evaluate(m, ds_train, ds_test) for name, m in models.items()}, names=["model", "turbine"])
scores["energy_dev_eur"] = scores["energy_dev_kwh"] / 1000 * PRICE_EUR_PER_MWH   # predicted - real revenue
scores.round(3)

wind_rmse  wind_mae  power_rmse  \
model                       turbine                                    
neighbour_mean              T1           1.380     1.065     820.693   
                            T2           1.117     0.873     693.197   
                            T3           1.183     0.920     738.470   
neighbour_mean_sector       T1           1.272     0.964     751.932   
                            T2           1.112     0.874     683.011   
                            T3           1.164     0.902     729.471   
ridge_all_references        T1           1.378     1.057     820.448   
                            T2           1.097     0.854     683.547   
                            T3           1.168     0.908     727.233   
ridge_all_references_sector T1           1.253     0.946     739.678   
                            T2           1.104     0.863     674.863   
                            T3           1.158     0.895     719.187   

                                     power_mae  energy_dev_pct  \
model                       turbine                              
neighbour_mean              T1         551.538          -8.626   
                            T2         465.094          -8.595   
                            T3         486.403         -12.835   
neighbour_mean_sector       T1         492.126         -10.384   
                            T2         461.564          -4.041   
                            T3         477.412         -10.812   
ridge_all_references        T1         545.508          -6.333   
                            T2         454.046          -6.033   
                            T3         479.426         -10.417   
ridge_all_references_sector T1         482.633          -8.541   
                            T2         453.732          -1.863   
                            T3         471.088          -9.195   

                                     energy_true_kwh  energy_pred_kwh  \
model                       turbine                                     
neighbour_mean              T1            756551.738       691294.561   
                            T2            918867.276       839889.826   
                            T3            783106.270       682596.043   
neighbour_mean_sector       T1            756551.738       677994.906   
                            T2            918867.276       881736.558   
                            T3            783106.270       698438.316   
ridge_all_references        T1            756551.738       708636.923   
                            T2            918867.276       863431.869   
                            T3            783106.270       701528.823   
ridge_all_references_sector T1            756551.738       691933.059   
                            T2            918867.276       901751.924   
                            T3            783106.270       711097.269   

                                     energy_dev_kwh  coverage  energy_dev_eur  
model                       turbine                                            
neighbour_mean              T1           -65257.177       1.0       -5220.574  
                            T2           -78977.450       1.0       -6318.196  
                            T3          -100510.228       1.0       -8040.818  
neighbour_mean_sector       T1           -78556.832       1.0       -6284.547  
                            T2           -37130.719       1.0       -2970.457  
                            T3           -84667.954       1.0       -6773.436  
ridge_all_references        T1           -47914.815       1.0       -3833.185  
                            T2           -55435.407       1.0       -4434.833  
                            T3           -81577.448       1.0       -6526.196  
ridge_all_references_sector T1           -64618.678       1.0       -5169.494  
                            T2           -17115.352       1.0       -1369.228  
                            T3          

In [12]:
# test period in energy and money, T1-T3 together: real vs predicted energy over the same time points
# (negative deviation = the model predicts less energy than the turbines really produced)
energy_test = scores[["energy_true_kwh", "energy_pred_kwh", "energy_dev_kwh", "energy_dev_eur"]].groupby(level="model", sort=False).sum()
energy_test["energy_dev_pct"] = 100 * energy_test["energy_dev_kwh"] / energy_test["energy_true_kwh"]
energy_test.round(1)

,energy_true_kwh,energy_pred_kwh,energy_dev_kwh,energy_dev_eur,energy_dev_pct
model,,,,,
neighbour_mean,2458525.3,2213780.4,-244744.9,-19579.6,-10.0
neighbour_mean_sector,2458525.3,2258169.8,-200355.5,-16028.4,-8.1
ridge_all_references,2458525.3,2273597.6,-184927.7,-14794.2,-7.5
ridge_all_references_sector,2458525.3,2304782.3,-153743.0,-12299.4,-6.3


In [13]:
# every model vs the first one (baseline): its errors, the difference (model - baseline) and the change in %;
# negative = smaller error = better
baseline = list(models)[0]
errors = scores[["wind_rmse", "wind_mae", "power_rmse", "power_mae"]].copy()
errors["abs_energy_dev_pct"] = scores["energy_dev_pct"].abs()   # size of the energy error, sign dropped
errors["abs_energy_dev_kwh"] = scores["energy_dev_kwh"].abs()
errors["abs_energy_dev_eur"] = scores["energy_dev_eur"].abs()

values = {}
for name in models:
    e = errors.loc[name].copy()
    e.loc["mean"] = e.mean()   # average over T1-T3
    values[name] = e.stack()
values = pd.DataFrame(values)   # rows (turbine, metric), one column per model
difference = values.drop(columns=baseline).sub(values[baseline], axis=0)
comparison = pd.concat({
    "value": values,
    "difference": difference,
    "change_%": 100 * difference.div(values[baseline], axis=0),
}, axis=1).swaplevel().reindex(errors.columns, level=0)
comparison.index.names = ["metric", "turbine"]
comparison.round(3)

value                        \
                           neighbour_mean neighbour_mean_sector   
metric             turbine                                        
wind_rmse          T1               1.380                 1.272   
                   T2               1.117                 1.112   
                   T3               1.183                 1.164   
                   mean             1.227                 1.183   
wind_mae           T1               1.065                 0.964   
                   T2               0.873                 0.874   
                   T3               0.920                 0.902   
                   mean             0.953                 0.913   
power_rmse         T1             820.693               751.932   
                   T2             693.197               683.011   
                   T3             738.470               729.471   
                   mean           750.787               721.471   
power_mae          T1             551.538               492.126   
                   T2             465.094               461.564   
                   T3             486.403               477.412   
                   mean           501.012               477.034   
abs_energy_dev_pct T1               8.626                10.384   
                   T2               8.595                 4.041   
                   T3              12.835                10.812   
                   mean            10.019                 8.412   
abs_energy_dev_kwh T1           65257.177             78556.832   
                   T2           78977.450             37130.719   
                   T3          100510.228             84667.954   
                   mean         81581.618             66785.168   
abs_energy_dev_eur T1            5220.574              6284.547   
                   T2            6318.196              2970.457   
                   T3            8040.818              6773.436   
                   mean          6526.529              5342.813   

                                                                             \
                           ridge_all_references ridge_all_references_sector   
metric             turbine                                                    
wind_rmse          T1                     1.378                       1.253   
                   T2                     1.097                       1.104   
                   T3                     1.168                       1.158   
                   mean                   1.214                       1.172   
wind_mae           T1                     1.057                       0.946   
                   T2                     0.854                       0.863   
                   T3                     0.908                       0.895   
                   mean                   0.940                       0.902   
power_rmse         T1                   820.448                     739.678   
                   T2                   683.547                     674.863   
                   T3                   727.233                     719.187   
                   mean                 743.743                     711.243   
power_mae          T1                   545.508                     482.633   
                   T2                   454.046                     453.732   
                   T3                   479.426                     471.088   
                   mean                 492.993                     469.151   
abs_energy_dev_pct T1                     6.333                       8.541   
                   T2                     6.033                       1.863   
                   T3                    10.417                       9.195   
                   mean                   7.594                       6.533   
abs_energy_dev_kwh T1                 47914.815                   64618.678   
                   T2                 55435.407               

## 5. Pick the best model

In [14]:
best_name = select_best(scores)   # lowest mean wind RMSE over T1-T3; select_best(scores, "energy_dev_pct") for energy
best_name

'ridge_all_references_sector'

## 6. Retrain on train + test, predict the outage

In [15]:
# retrain the best model on more data, including the days right after the outage
ds_full = pd.concat([ds_train, ds_test]).sort_index()
ds_full = ds_full[ds_full["complete"]]   # same row filter as use_non_complete=False
results_sub = models[best_name](ds_full, ds_result)

print(f"outage {ds_result.index[0]} -> {ds_result.index[-1]}, missing predictions: {results_sub.isna().sum().sum()}")
results_sub.describe().round(2)

outage 2026-09-01 00:00:00 -> 2026-09-03 23:50:00, missing predictions: 0


,T1,T2,T3
count,432.00,432.00,432.00
mean,5.87,6.70,6.16
std,1.39,1.56,1.38
min,3.04,3.10,2.94
25%,4.75,5.34,4.97
50%,5.88,6.82,6.28
75%,6.60,7.84,7.13
max,9.54,9.99,9.36


## 7. Energy loss and submission

In [16]:
loss = pd.DataFrame({
    "mean_wind_ms": results_sub.mean(),
    "lost_kwh": pd.Series({t: energy_kwh(results_sub[t]) for t in results_sub}),
})
loss["lost_mwh"] = loss["lost_kwh"] / 1000
loss["lost_eur"] = loss["lost_mwh"] * PRICE_EUR_PER_MWH
loss.loc["Total", ["lost_kwh", "lost_mwh", "lost_eur"]] = loss[["lost_kwh", "lost_mwh", "lost_eur"]].sum()
loss.round(1)

,mean_wind_ms,lost_kwh,lost_mwh,lost_eur
T1,5.9,82605.6,82.6,6608.5
T2,6.7,121822.3,121.8,9745.8
T3,6.2,94053.2,94.1,7524.3
Total,NaN,298481.1,298.5,23878.5


In [17]:
# lost energy and revenue during the outage per model, each retrained on train + test like the best model above
baseline = list(models)[0]
lost_mwh = {}
for name, model in models.items():
    pred = model(ds_full, ds_result)
    per_turbine = pd.Series({t: energy_kwh(pred[t]) / 1000 for t in pred})
    lost_mwh[name] = pd.concat([per_turbine, pd.Series({"Total": per_turbine.sum()})])
lost_mwh = pd.DataFrame(lost_mwh)

loss_by_model = pd.concat({"lost_mwh": lost_mwh, "lost_eur": lost_mwh * PRICE_EUR_PER_MWH}, axis=1)
for name in list(models)[1:]:   # difference of every other model to the first one
    loss_by_model["lost_mwh", f"{name} - {baseline}"] = lost_mwh[name] - lost_mwh[baseline]
    loss_by_model["lost_eur", f"{name} - {baseline}"] = (lost_mwh[name] - lost_mwh[baseline]) * PRICE_EUR_PER_MWH
loss_by_model[["lost_mwh", "lost_eur"]].round(1)

lost_mwh                                             \
      neighbour_mean neighbour_mean_sector ridge_all_references   
T1              87.6                  82.0                 94.0   
T2             110.8                 117.3                117.8   
T3              89.4                  94.1                 93.8   
Total          287.8                 293.4                305.6   

                                                                          \
      ridge_all_references_sector neighbour_mean_sector - neighbour_mean   
T1                           82.6                                   -5.6   
T2                          121.8                                    6.5   
T3                           94.1                                    4.7   
Total                       298.5                                    5.5   

                                             \
      ridge_all_references - neighbour_mean   
T1                                      6.4   
T2                                      7.1   
T3                                      4.4   
Total                                  17.8   

                                                         lost_eur  \
      ridge_all_references_sector - neighbour_mean neighbour_mean   
T1                                            -5.0         7009.4   
T2                                            11.0         8862.7   
T3                                             4.6         7154.6   
Total                                         10.6        23026.7   

                                                                              \
      neighbour_mean_sector ridge_all_references ridge_all_references_sector   
T1                   6557.4               7519.4                      6608.5   
T2                   9381.1               9427.2                      9745.8   
T3                   7530.6               7505.3                      7524.3   
Total               23469.2              24451.9                     23878.5   

                                              \
      neighbour_mean_sector - neighbour_mean   
T1                                    -451.9   
T2                                     518.5   
T3                                     376.0   
Total                                  442.5   

                                             \
      ridge_all_references - neighbour_mean   
T1                                    510.0   
T2                                    564.5   
T3                                    350.7   
Total                                1425.3   

                                                    
      ridge_all_references_sector - neighbour_mean  
T1                                          -400.9  
T2                                           883.1  
T3                                           369.6  
Total                                        851.8

In [18]:
# TODO: confirm the submission format with the organisers
# results_sub.to_csv("submission.csv")